In [ ]:
# THIS BLOCK IS FROM MODULE 4 - DO NOT DELETE OR MODIFY
# AND IS THE BASIS FOR THE MODULE 9 PROSTGRESQL DASHBOARD

# cs590-chapter4.ipynb - This is a Jupyter Notebook designed to 
# connect to a PostgreSQL database and provide CRUD operations.
#
# This notebook uses a supplemental configuration file that has the
# following format:
# 
# [postgresql]
# host=localhost
# database=Nutrition
# user=cs590
# password=REPLACE_WITH_YOUR_PASSWORD_FOR_THE_DATABASE_USER
#
# We also build a config function to read the data from the
# configuration file to make the code easier to maintain and more
# secure. 
#
# Remember: You should _NEVER_ embed a password in your 
# source code!
#
#------------------------------------------------------------------
# Change History
#------------------------------------------------------------------
# Version   |   Description
#------------------------------------------------------------------
#    1          Initial Development
#------------------------------------------------------------------

import psycopg2
from configparser import ConfigParser

#
# config - This function is designed to read the data from our
# configuration file to allow us to setup the connection to our
# PostgreSQL database.
#
# Please Note: We create this method genericly so that it can
# be reused with other configuration data separated by section
# as key-value pairs
#
# Parameters:
#
#   filename: The name of the file that contains the configuration data
#   section:  The name of the section within the configuration file
#             for which we are retrieving data
# 
# Returns:
#   A dictionary with the key-value pairs from the configuration file.
#
def config(filename, section):
    # Initialize a configuration parser
    cp = ConfigParser()
    
    # Acquire data from the configuration file
    cp.read(filename)
    
    # Import configuration section as a dictionary
    block = {}
    
    # Check to make sure that the section exists in our config file
    if cp.has_section(section):
        # Read the entire section
        items = cp.items(section)
        
        # Loop through config items to create the dictionary
        for item in items:
            block[item[0]] = item[1]
    else:
        # If we have gotten here the section did not exist in the
        # config file, so we raise an exception
        raise Exception(f"Section {section} was not found in the {filename} file!")
    
    # Return the dictionary to the calling program
    return block


#
# displayConfig - This function is designed to test the config function.
# This function will call the config function with a filename and a section
# obtain the resulting data, and then print the retrieved information to
# the screen. This can be useful for debugging and testing purposes.
#
# Parameters:
#
#   filename: The name of the file that contains the configuration data
#   section:  The name of the section within the configuration file
#             for which we are retrieving data
# 
def displayConfig(filename, section):
    # Obtain configuration data from the config file
    confData = config(filename, section)
    
    # Display results
    print(confData)
    

#
# dbConnect - function designed to connect to the database. This function
# leverages the config method to pull the necessary data from our database
# configuration file.
#
# Parameters:
#
#   filename: The name of the file that contains the configuration data
#   section:  The name of the section within the configuration file
#             for which we are retrieving data
# 
# Returns a Connection to the databse or null in the event of a problem
#
def dbConnect(filename, section):
    # Obtain configuration data from the config file
    confData = config(filename, section)

    # create connection variable
    conn = None

    # Create a try/except block for the connection. This is important
    # in order to keep from blowing up the application in the event that
    # we cannot connect to the database.
    try:     
        # Setup the connection to the PostgreSQL server
        conn = psycopg2.connect(
            dbname = confData["database"],
            user = confData["user"],
            password = confData["password"],
            host = confData["host"]
        )

        # Return the database connection for use in the application
        # Don't forget to close the connection when done.
        return conn
    except(Exception, psycopg2.DatabaseError) as error:
        # We had a problem, so lets see what it was
        print(error)
        return None
    
#
# dbTest - a utility method to test connection to the database
# This method will connect to the database with the information from
# the configuration file, validate that the connection was made to the 
# Appropriate database, print the result, and then close the database
# connection. This function is useful for debugging initial connections. 
# 
# Parameters:
#
#   filename: The name of the file that contains the configuration data
#   section:  The name of the section within the configuration file
#             for which we are retrieving data
#
def dbTest(filename, section):
    # create our connection
    print('Connecting to our PostgreSQL database...')
    conn = dbConnect(filename, section)

    # Create a try/except block for the connection. This is important
    # in order to keep from blowing up the application in the event that
    # we cannot connect to the database.
    try:     
        # Create a cursor
        cursor = conn.cursor()

        # test the connection
        cursor.execute('SELECT current_database()')

        # display result
        print(cursor.fetchone())

        # close the cursor
        cursor.close()
    except(Exception, psycopg2.DatabaseError) as error:
        # We had a problem, so lets see what it was
        print(error)
    finally:
        # Everything worked well, so we need to close the connection
        if conn is not None:
            conn.close()


#
# create - CRUD Method used to create a new record in a database. 
# This method will be used to create a new record to be added to a database.
# Because PostgreSQL is a relational database, this will add a record to a
# single table.
#
# Parameters:
#     query: SQL statement to insert record into the appropriate database table
#     conn: The database connection to use create the record
#
def createRecord(query, conn):
    # TODO: Create the code necessary to perform a create operation
    # against your PostgreSQL database.
    #
    # Your code must use the function signature listed above and it
    # must provide the correct test output when this notebook is run.
    try:
        
        cursor = conn.cursor()
        cursor.execute(query)
        conn.commit()
        cursor.close()
        print("create query executed...")
    except(Exception, psycopg2.DatabaseError) as error:
        print(error)

        if conn is not None:
            conn.rollback()
    
    
                       

#
# testCreateRecord - Method used to test the createRecord function.
# This function will exercise the createRecord function by attempting to add
# a record to the Main (Nutrient description) table. This function will
# only work with the Nutrient database, it is not portable.
#
# Parameters:
#   filename: The name of the file that contains the configuration data
#   section:  The name of the section within the configuration file
#             for which we are retrieving data
#
def testCreateRecord(filename, section):
    # create our connection
    print('Connecting to our PostgreSQL database...')
    conn = dbConnect(filename, section)

    # Setup the SQL Statement

    if conn is None:
        print("skipping...")
        return
    

    sqlSttm = """INSERT INTO public.Main VALUES ( 'Feature',6.6 );"""

    query = sqlSttm

    createRecord(query, conn)

    if conn is not None:
        conn.close()
        return
    #conn.close()


#
# readRecord - CRUD Method used to read a record from a database. 
# This method will be used to read a new record from a database table.
# Because PostgreSQL is a relational database, this will read a record from a
# single table.
#
# Parameters:
#     tablename: The table to select the record(s) from
#     clause: the appropriate clause to test for - specifically what you would 
#             see after the SQL WHERE clause
#     conn: The database connection to use create the record
#    
def readRecord(tablename, clause, conn):
    # TODO: Create the code necessary to perform a read operation
    # against your PostgreSQL database.
    #
    # Your code must use the function signature listed above and it
    # must provide the correct test output when this notebook is run.
    try:
        cursor = conn.cursor()
        cursor.execute(f"SELECT * FROM {tablename} WHERE {clause}")
        records = cursor.fetchall()
        cursor.close()
        print("read query executed")
        return records
    except(Exception, psycopg2.DatabaseError) as error:
        print(error)
        if conn is not None:
            conn.rollback()
            return None
        
    
    


#
# testReadRecord - Method used to test the readRecord function.
# This function will exercise the readRecord function by attempting to read
# a record from the Main (Nutrient description) table. This function will
# only work with the Nutrient database, it is not portable.
#
# Parameters:
#   filename: The name of the file that contains the configuration data
#   section:  The name of the section within the configuration file
#             for which we are retrieving data
#
def testReadRecord(filename, section):
    # create our connection
    print('Connecting to our PostgreSQL database...')
    conn = dbConnect(filename, section)

    # Setup the SQL Statement
    records = readRecord("Main",'"properties/mag" = 6.6', conn)

    # Display results
    # print(f"There are {len(records)} records in the result set.")
    #for r in records:
    #    print(r)
    print(records)
    
    conn.close()

#
# updateRecord - CRUD Method used to update a record from a database. 
# This method will be used to update a record from a database table.
# Because PostgreSQL is a relational database, this will update a record from a
# single table.
#
# Parameters:
#     tablename: The table to update the record in
#     clause: The update clause - what you would normally see after the SQL SET clause
#     conn: The database connection to use create the record
#    
def updateRecord(tablename, clause, conn):
    # TODO: Create the code necessary to perform an update operation
    # against your PostgreSQL database.
    #
    # Your code must use the function signature listed above and it
    # must provide the correct test output when this notebook is run.
    try:
        
        cursor = conn.cursor()
        cursor.execute(f"UPDATE public.{tablename} SET {clause}")
        conn.commit()
        cursor.close()
        print("update query executed")
    except(Exception, psycopg2.DatabaseError) as error:
        print(error)
        if conn is not None:
            conn.rollback()
            return None
    
    


#
# testUpdateRecord - Method used to test the updateRecord function.
# This function will exercise the updateRecord function by attempting to update
# a record from the Main (Nutrient description) table. This function will
# only work with the Nutrient database, it is not portable.
#
# Parameters:
#   filename: The name of the file that contains the configuration data
#   section:  The name of the section within the configuration file
#             for which we are retrieving data
#
def testUpdateRecord(filename, section):
    # create our connection
    print('Connecting to our PostgreSQL database...')
    conn = dbConnect(filename, section)

    # Setup the SQL Statement
    updateRecord("Main", """"geometry/coordinates2" = '76.2' where "geometry/coordinates1" = 60.5778""", conn)    
    conn.close()


#
# deleteRecord - CRUD Method used to delete a record from a database. 
# This method will be used to delete a record from a database table.
# Because PostgreSQL is a relational database, this will update a record from a
# single table.
#
# NOTE: USE WITH CAUTION! THIS METHOD WILL DESTROY DATA
#
# Parameters:
#     tablename: The table to update the record in
#     clause: The update clause - what you would normally see after the SQL WHERE clause
#     conn: The database connection to use create the record
#    
def deleteRecord(tablename, clause, conn):
    # TODO: Create the code necessary to perform a delete operation
    # against your PostgreSQL database.
    #
    # Your code must use the function signature listed above and it
    # must provide the correct test output when this notebook is run.
    try:
        
        cursor = conn.cursor()
        cursor.execute(f"DELETE FROM {tablename} WHERE {clause}")
        conn.commit()
        cursor.close()
        print("delete query executed")
    except(Exception, psycopg2.DatabaseError) as error:
        print(error)
        if conn is not None:
            conn.rollback()
            return None
    
    #raise NotImplementedError


#
# testDeleteRecord - Method used to test the deleteRecord function.
# This function will exercise the deleteRecord function by attempting to remove
# a record from the Main (Nutrient description) table. This function will
# only work with the Nutrient database, it is not portable.
#
# Parameters:
#   filename: The name of the file that contains the configuration data
#   section:  The name of the section within the configuration file
#             for which we are retrieving data
#
def testDeleteRecord(filename, section):
    # create our connection
    print('Connecting to our PostgreSQL database...')
    conn = dbConnect(filename, section)

    # Setup the SQL Statement
    records = deleteRecord("Main",""" "properties/mag" = 6.6 """, conn)    
    conn.close()

def unix_timestamp_to_datetime(filename, section):#for converting properties/time to datetime format, currently stored as int8 in the database with commas : ()
    from datetime import datetime
    conn = dbConnect(filename, section)
    cursor = conn.cursor()
    cursor.execute("alter table \"public.Main\", public.properties, public.features alter column \"properties/time\" type timestamp using to_timestamp(\"properties/time\"::double precision / 1000);")
    result = cursor.fetchone()
    cursor.close()
    conn.close()
    return result[0] if result else None

# if __name__ == '__main__':
#     displayConfig('database.conf','postgresql')
#     dbTest('database.conf','postgresql')
#     testCreateRecord('database.conf','postgresql')
#     testReadRecord('database.conf','postgresql')
#     testUpdateRecord('database.conf','postgresql')
#     testDeleteRecord('database.conf','postgresql')

In [ ]:
# THIS BLOCK IS THE START OF THE MODULE 9 DASHBOARD - DO NOT DELETE OR MODIFY
import dash
import pandas as pd
import plotly.express as px
from dash import Dash
from dash import dcc, html, Input, Output, dash_table


app = Dash()


app.layout = html.Div([
    html.H1("Module 9 PostgreSQL Dashboard"),
    html.Div([
        html.Label("Select a Table:"),
        dcc.Dropdown(
            id='table-dropdown',
            options=[
                {'label': 'Main', 'value':  'Main'},
                {'label': 'properties', 'value': 'properties'},
                {'label': 'geometry', 'value': 'geometry'},
                {'label': 'features', 'value': 'features'}
            ],
            value= 'Main'
        ),
        dcc.Graph(id='table-graph'),
        dcc.Input(id='table-input', type='text', placeholder='Enter a value to filter the table...'),
        dash_table.DataTable(id='table-data', columns=[{"name": i, "id": i} for i in pd.read_sql("SELECT * FROM \"Main\" LIMIT 1", dbConnect('database.conf', 'postgresql')).columns]),
        dcc.Input(
        id='search-bar',
        type='text',
        placeholder='Type to search...',
        debounce=False  # Updates on every keystroke if False
    ),
    html.Div(id='search-output'),
    #line graph to display earthquakes over time, from properties/type and properties/time
    #also need to convert the properties/time from unix timestamp to datetime format for the x-axis, unit is ms and UTC timezone, so need to use pd.to_datetime with unit='ms' and utc=True

    dcc.Graph(id='earthquake-graph', figure=px.line(pd.read_sql("SELECT \"properties/time\", \"properties/mag\" FROM \"Main\" ORDER BY \"properties/time\" LIMIT 100", dbConnect('database.conf', 'postgresql')), x="properties/time", y="properties/mag", title="Earthquakes Over Time"))
        
    ])])

@app.callback(
        Output('table-data', 'data'),
        Input('search-bar', 'value')
    )
def update_output(search_value):
        if search_value:
            return update_table(search_value)
        else:
            return pd.read_sql("SELECT * FROM \"Main\" LIMIT 10", dbConnect('database.conf', 'postgresql')).to_dict('records')    

    
def update_table(search_value):
        # Connect to the database
        conn = dbConnect('database.conf', 'postgresql')
        
        # Query the table with search filter
        query = f"SELECT * FROM \"Main\" WHERE column_name ILIKE '%{search_value}%'"
        df = pd.read_sql(query, conn)
        
        # Close the connection
        conn.close()
        
        return df.to_dict('records')

@app.callback(
        Output('table-graph', 'figure'),
        Input('table-dropdown', 'value')
    )
def update_graph(selected_table):
        # Connect to the database
        conn = dbConnect('database.conf', 'postgresql')
        
        # Query the selected table
        query = f"SELECT * FROM {selected_table} LIMIT 10"
        df = pd.read_sql(query, conn)
        
        # Close the connection
        conn.close()
        
        # Create a bar chart of the first numeric column
        numeric_cols = df.select_dtypes(include=['number']).columns
        if len(numeric_cols) > 0:
            fig = px.bar(df, x=df.index, y=numeric_cols[0], title=f"Top 10 Records from {selected_table}")
            return fig
        else:
            return px.bar(title=f"No numeric columns in {selected_table}")


#app = jupyter_dash.JupyterDash(__name__)

if __name__ == '__main__':
    unix_timestamp_to_datetime('database.conf', 'postgresql')
    app.run(jupyter_mode='inline', debug=True)


/var/folders/4_/x2kyvt3d7tl304c5lc3p4d6h0000gn/T/ipykernel_80562/252596699.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dash_table.DataTable(id='table-data', columns=[{"name": i, "id": i} for i in pd.read_sql("SELECT * FROM \"Main\" LIMIT 1", dbConnect('database.conf', 'postgresql')).columns]),
/var/folders/4_/x2kyvt3d7tl304c5lc3p4d6h0000gn/T/ipykernel_80562/252596699.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dcc.Graph(id='earthquake-graph', figure=px.line(pd.read_sql("SELECT \"properties/time\", \"properties/mag\" FROM \"Main\" ORDER BY \"properties/time\" LIMIT 100", dbConnect('database.conf', 'postgresql')), x="properties/time", y="properties/mag", title="

/var/folders/4_/x2kyvt3d7tl304c5lc3p4d6h0000gn/T/ipykernel_80562/252596699.py:74: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
/var/folders/4_/x2kyvt3d7tl304c5lc3p4d6h0000gn/T/ipykernel_80562/252596699.py:48: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql("SELECT * FROM \"Main\" LIMIT 10", dbConnect('database.conf', 'postgresql')).to_dict('records')
/var/folders/4_/x2kyvt3d7tl304c5lc3p4d6h0000gn/T/ipykernel_80562/252596699.py:74: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  